数据分析顺序：

体检 → 去重 → 标准化 → 空值 → 类型 → 派生 → 前后对比。顺序不是死规定，但每步做完要能解释为什么放在这个位置

1、读取与体检

In [3]:
import pandas as pd

In [33]:
# 读取描述
df = pd.read_csv('events.csv' , encoding='utf-8')
df.head(10)

,event_id,user_id,event_time,event_type,channel,device,amount
0,E000001,U0001,2026-09-29 23:22:00,exposure,App,iOS,0.0
1,E000002,U0001,2026-09-15 21:43:00,exposure,App,iOS,0.0
2,E000003,U0001,2026-10-05 05:16:00,exposure,App,iOS,0.0
3,E000004,U0001,2026-09-19 23:07:00,exposure,App,iOS,0.0
4,E000005,U0001,2026-10-09 06:23:00,exposure,App,iOS,0.0
5,E000006,U0001,2026-10-09 06:41:00,click,App,iOS,0.0
6,E000007,U0001,2026-09-21 15:41:00,exposure,App,iOS,0.0
7,E000008,U0001,2026-09-19 09:25:00,exposure,App,iOS,0.0
8,E000009,U0001,2026-09-19 09:48:00,click,App,iOS,0.0
9,E000010,U0001,2026-09-19 10:40:00,add_cart,App,iOS,0.0


In [34]:
# 表格形状
df.shape

(21162, 7)

In [35]:
# 表格各列类型
df.dtypes

event_id       object
user_id        object
event_time     object
event_type     object
channel        object
device         object
amount        float64
dtype: object

In [36]:
# 观察表结构如何
df.info

<bound method DataFrame.info of       event_id user_id           event_time event_type channel   device  \
0      E000001   U0001  2026-09-29 23:22:00   exposure     App      iOS   
1      E000002   U0001  2026-09-15 21:43:00   exposure     App      iOS   
2      E000003   U0001  2026-10-05 05:16:00   exposure     App      iOS   
3      E000004   U0001  2026-09-19 23:07:00   exposure     App      iOS   
4      E000005   U0001  2026-10-09 06:23:00   exposure     App      iOS   
...        ...     ...                  ...        ...     ...      ...   
21157  E014313   U1377  2026-09-15 07:58:00   exposure     App  Android   
21158  E019564   U1905  2026-09-21 02:26:00   exposure     小程序      Web   
21159  E015617   U1507  2026-10-23 10:45:00   exposure     小程序      Web   
21160  E017262   U1667  2026-09-11 20:02:00      click     App      Web   
21161  E012264   U1153  2026-09-24 13:54:00   add_cart     小程序  Android   

       amount  
0         0.0  
1         0.0  
2         0.0  
3  

In [37]:
# 判断数值列的分布怎么样
df.describe()

,amount
count,21162.000000
mean,37.306461
std,219.437086
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,1997.750000


In [38]:
# 检查表格列所有缺失值数量
df.isna().sum()

event_id      0
user_id       0
event_time    0
event_type    0
channel       0
device        0
amount        0
dtype: int64

In [39]:
# 事件类型分布
df['event_type'].value_counts(dropna=False)

event_type
exposure    12840
click        4443
add_cart     2027
order         974
pay           764
Click          93
PAY            21
Name: count, dtype: int64

In [40]:
# 渠道分布
df['channel'].value_counts(dropna = False)

channel
App    10141
小程序     6536
PC      4194
app      206
pc        85
Name: count, dtype: int64

In [41]:
# 金额描述
df['amount'].describe()

count    21162.000000
mean        37.306461
std        219.437086
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max       1997.750000
Name: amount, dtype: float64

2、去重

In [42]:
before = len(df)
dup_mask = df.duplicated(subset=["event_id"], keep=False)
print("重复的 event_id 行数:", dup_mask.sum())
print(df[dup_mask].sort_values("event_id").head(10))

df = df.drop_duplicates(subset=["event_id"], keep="first").copy()
after = len(df)
print(f"去重：{before} → {after}，删除 {before - after} 行")

重复的 event_id 行数: 1237
      event_id user_id           event_time event_type channel   device  \
35     E000036   U0005  2026-10-13 09:24:00   exposure     App      iOS   
20814  E000036   U0005  2026-10-13 09:24:00   exposure     app      iOS   
72     E000073   U0009  2026-09-30 00:06:00   exposure     小程序      iOS   
20870  E000073   U0009  2026-09-30 00:06:00   exposure     小程序      iOS   
76     E000077   U0009  2026-09-25 00:44:00   exposure     小程序      iOS   
21144  E000077   U0009  2026-09-25 00:44:00   exposure     小程序      iOS   
82     E000083   U0010  2026-09-24 18:55:00   exposure     小程序  Android   
20592  E000083   U0010  2026-09-24 18:55:00   exposure     小程序  Android   
88     E000089   U0012  2026-09-20 11:17:00   exposure      PC      iOS   
20859  E000089   U0012  2026-09-20 11:17:00   exposure      pc      iOS   

       amount  
35        0.0  
20814     0.0  
72        0.0  
20870     0.0  
76        0.0  
21144     0.0  
82        0.0  
20592     0.0  
88      

3、统一事件类型

In [43]:
# 统一事件类型
print("处理前:", df["event_type"].value_counts(dropna=False).to_dict())

df["event_type"] = (
    df["event_type"]
    .astype(str)
    .str.strip()      # 去首尾空格
    .str.lower()      # 统一小写
)

# 用映射表兜底：把非标准值显式映射，映射不到的标为 unknown
valid = {"exposure", "click", "add_cart", "order", "pay"}
df["event_type"] = df["event_type"].where(df["event_type"].isin(valid), "unknown")

print("处理后:", df["event_type"].value_counts(dropna=False).to_dict())

处理前: {'exposure': 12469, 'click': 4408, 'add_cart': 1959, 'order': 947, 'pay': 758}
处理后: {'exposure': 12469, 'click': 4408, 'add_cart': 1959, 'order': 947, 'pay': 758}


4、统一渠道

In [44]:
# 统一渠道
print("处理前:", df["channel"].value_counts(dropna=False).to_dict())

channel_map = {
    "app": "App", "APP": "App", "App": "App",
    "小程序": "小程序", "mini": "小程序",
    "pc": "PC", "PC": "PC", "web": "PC",
}
df["channel"] = (
    df["channel"].astype(str).str.strip()
    .replace(channel_map)
)
df["channel"] = df["channel"].where(df["channel"].isin({"App", "小程序", "PC"}), "unknown")

print("处理后:", df["channel"].value_counts(dropna=False).to_dict())

处理前: {'App': 10032, '小程序': 6356, 'PC': 4153}
处理后: {'App': 10032, '小程序': 6356, 'PC': 4153}


5、处理时间空值

In [45]:
before = len(df)
df["event_time"] = pd.to_datetime(df["event_time"], errors="coerce")

na_time = df["event_time"].isna()
print("时间空值/非法行数:", na_time.sum())
print(df.loc[na_time, ["event_id", "user_id", "event_type"]].head())

# 策略：删除无法解析时间的行（因为无法参与时间分析），并记录
df = df.dropna(subset=["event_time"]).copy()
print(f"时间清洗：{before} → {len(df)}，删除 {before - len(df)} 行")

时间空值/非法行数: 0
Empty DataFrame
Columns: [event_id, user_id, event_type]
Index: []
时间清洗：20541 → 20541，删除 0 行


6、时间字段派生

In [46]:
df["date"] = df["event_time"].dt.date
df["hour"] = df["event_time"].dt.hour
df["dow"] = df["event_time"].dt.dayofweek        # 0=周一, 6=周日
df["dow_name"] = df["dow"].map({
    0:"周一",1:"周二",2:"周三",3:"周四",4:"周五",5:"周六",6:"周日"
})
df["week"] = df["event_time"].dt.to_period("W").astype(str)
df["month"] = df["event_time"].dt.to_period("M").astype(str)

print(df[["event_time","date","hour","dow","dow_name","week","month"]].head())

           event_time        date  hour  dow dow_name                   week  \
0 2026-09-29 23:22:00  2026-09-29    23    1       周二  2026-09-28/2026-10-04   
1 2026-09-15 21:43:00  2026-09-15    21    1       周二  2026-09-14/2026-09-20   
2 2026-10-05 05:16:00  2026-10-05     5    0       周一  2026-10-05/2026-10-11   
3 2026-09-19 23:07:00  2026-09-19    23    5       周六  2026-09-14/2026-09-20   
4 2026-10-09 06:23:00  2026-10-09     6    4       周五  2026-10-05/2026-10-11   

     month  
0  2026-09  
1  2026-09  
2  2026-10  
3  2026-09  
4  2026-10  


7、类型转换

In [52]:
df["amount"] = pd.to_numeric(df["amount"], errors="coerce").fillna(0.0)
df["user_id"] = df["user_id"].astype(str)
df["event_id"] = df["event_id"].astype(str)
df["event_type"] = df["event_type"].astype("category")
df["channel"] = df["channel"].astype("category")

print(df.dtypes)
print("amount 非数值行数:", df["amount"].isna().sum())

event_id              object
user_id               object
event_time    datetime64[ns]
event_type          category
channel             category
device                object
amount               float64
date                  object
hour                   int32
dow                    int32
dow_name              object
week                  object
month                 object
dtype: object
amount 非数值行数: 0


In [60]:
df["user_id"].nunique()

1791

In [57]:
df.to_csv('events_clean.csv' , index = False,encoding='utf-8')

In [61]:
df1 = pd.read_csv(
    "events_clean.csv",
    encoding="utf-8",
    parse_dates=["event_time"],          # 时间列直接解析成 datetime
    dtype={
        "user_id": str,
        "event_id": str,
        "event_type": "category",
        "channel": "category",
    },
)

print(df1.dtypes)

event_id              object
user_id               object
event_time    datetime64[ns]
event_type          category
channel             category
device                object
amount               float64
date                  object
hour                   int64
dow                    int64
dow_name              object
week                  object
month                 object
dtype: object


In [62]:
df1.shape

(20541, 13)

In [63]:
df1.dtypes

event_id              object
user_id               object
event_time    datetime64[ns]
event_type          category
channel             category
device                object
amount               float64
date                  object
hour                   int64
dow                    int64
dow_name              object
week                  object
month                 object
dtype: object

In [64]:
df1.isna().sum()

event_id      0
user_id       0
event_time    0
event_type    0
channel       0
device        0
amount        0
date          0
hour          0
dow           0
dow_name      0
week          0
month         0
dtype: int64